# 03 — Medication Timeline

**PrioMed-Rec | Sub-Task 3**

Builds the unified patient medication timeline and splits it into:
- `pre_admission_meds` — medications active BEFORE encounter_start
- `admission_meds` — medications during the encounter window

**Leakage prevention:** No post-admission data enters pre-admission features.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.data.admission_cohort import build_admission_cohort
from src.data.medication_timeline import (
    build_medication_timeline, build_pre_admission_meds,
    build_admission_meds, save_timeline_artifacts
)

## 1. Load cohort

In [ ]:
cohort = build_admission_cohort('../data/raw/synthea')
print(f'Cohort: {len(cohort):,} rows')

## 2. Build unified medication timeline

In [ ]:
timeline = build_medication_timeline('../data/raw/synthea')
print(f'Timeline: {len(timeline):,} rows')
print('Source breakdown:')
print(timeline['source'].value_counts().to_string())
timeline.head(3)

## 3. Pre-admission medications (leakage-safe)

In [ ]:
pre_meds = build_pre_admission_meds(cohort, timeline)
print(f'Pre-admission meds: {len(pre_meds):,} rows')
print(f'Unique admissions covered: {pre_meds["admission_encounter_id"].nunique():,}')
print('\nStatus distribution:')
print(pre_meds['status'].value_counts().to_string())
pre_meds.head(3)

## 4. Admission medications

In [ ]:
adm_meds = build_admission_meds(cohort, timeline)
print(f'Admission meds: {len(adm_meds):,} rows')
print(f'Unique encounters covered: {adm_meds["encounter_id"].nunique():,}')
adm_meds.head(3)

## 5. Leakage verification

In [ ]:
# Verify no pre-admission event is >= encounter_start
leaks = pre_meds[pre_meds['event_datetime'] >= pre_meds['encounter_start']]
print(f'Future leaks in pre_admission_meds: {len(leaks)} (expected: 0)')
assert len(leaks) == 0, 'LEAKAGE DETECTED'

## 6. Save all artifacts

In [ ]:
save_timeline_artifacts(timeline, pre_meds, adm_meds, '../data/interim')
print('Done.')